# ⚙️ Storage Optimization & Specialized Indexes

This notebook demonstrates pgVectorDB's storage-saving features:

| Feature | Savings | Method |
|:--------|:--------|:-------|
| Half-precision (float16) | 50% | `create_halfvec_table()` |
| Binary quantization | 87.5% | `build_index_binary_quantized()` |
| Subvector/Matryoshka | Variable | `build_index_with_subvectors()` |
| Sparse vectors | High-dim | `create_sparsevec_table()` |

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from langchain_core.documents import Document

from pgvectordb import Config, IndexType, pgVectorDB

## 1. Half-Precision Vectors (float16)

Standard vectors use 4 bytes per dimension (float32). Halfvec cuts that to 2 bytes — **50% storage savings** with minimal accuracy loss.

In [2]:
rag_fp16 = pgVectorDB(
    collection_name="nb_halfvec",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag_fp16.initialize(overwrite_existing=True)

# Create a half-precision table
await rag_fp16.create_halfvec_table(table_name="nb_halfvec")
print("✅ Half-precision table created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Half-precision table created


In [3]:
docs = [
    Document(page_content="Quantum computing uses qubits and superposition."),
    Document(page_content="Deep learning requires massive GPU compute."),
    Document(page_content="Float16 (half precision) saves memory during inference."),
]

await rag_fp16.add_documents(docs)

# Search — works exactly the same as float32
results = await rag_fp16.query("What saves memory?").semantic().limit(2).to_list()
print("🔍 Halfvec search results:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content']}")

🔍 Halfvec search results:
  [0.5480] Float16 (half precision) saves memory during inference.
  [0.7313] Quantum computing uses qubits and superposition.


In [4]:
await rag_fp16.delete_table()
await rag_fp16.close()
print("✅ Section 1 cleaned up")

✅ Section 1 cleaned up


## 2. Binary Quantization (87.5% savings)

Each float dimension is compressed to a **single bit** (positive → 1, negative → 0). Search uses Hamming distance, then reranks top candidates with full-precision cosine.

In [5]:
rag_bin = pgVectorDB(
    collection_name="nb_binary",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag_bin.initialize(overwrite_existing=True)

await rag_bin.add_documents(docs)

# Build binary index
await rag_bin.build_index_binary_quantized(m=16, ef_construction=64)
print("✅ Binary quantized index built")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Binary quantized index built


In [6]:
# Two-stage search: Hamming → Cosine rerank
results = await rag_bin.search_with_binary_rerank(
    query="What saves memory?",
    k=2,
    rerank_top=3,  # fetch 3 via Hamming, rerank to top 2 via cosine
)

print("🔍 Binary reranked results:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content']}")

🔍 Binary reranked results:
  [1.0000] Float16 (half precision) saves memory during inference.
  [0.5000] Quantum computing uses qubits and superposition.


In [7]:
await rag_bin.delete_table()
await rag_bin.close()
print("✅ Section 2 cleaned up")

✅ Section 2 cleaned up


## 3. Subvector / Matryoshka Search

Index only the **first N dimensions** of each embedding. Search fast on N dims, then rerank with full dimensions.

> Works best with Matryoshka-trained models (e.g. OpenAI `text-embedding-3`).

In [8]:
rag_sub = pgVectorDB(
    collection_name="nb_subvector",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag_sub.initialize(overwrite_existing=True)

await rag_sub.add_documents(docs)

# Index only first 128 of 384 dimensions
SUBVEC_DIMS = 128
await rag_sub.build_index_with_subvectors(
    subvector_dims=SUBVEC_DIMS,
    index_type=IndexType.HNSW,
)
print(f"✅ Subvector index built (first {SUBVEC_DIMS} of 384 dims)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Subvector index built (first 128 of 384 dims)


In [9]:
# Two-stage: subvector → full rerank
results = await rag_sub.search_with_subvector_rerank(
    query="How to reduce compute costs?",
    k=2,
    subvector_dims=SUBVEC_DIMS,
)

print("🔍 Subvector reranked results:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content']}")

🔍 Subvector reranked results:
  [0.4163] Deep learning requires massive GPU compute.
  [0.2917] Float16 (half precision) saves memory during inference.


In [10]:
await rag_sub.delete_table()
await rag_sub.close()
print("✅ Section 3 cleaned up")

✅ Section 3 cleaned up


## 4. Concurrent Index Building (Zero Downtime)

In production, use `build_index_concurrent` to avoid locking the table during index creation.

In [11]:
rag_conc = pgVectorDB(
    collection_name="nb_concurrent",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag_conc.initialize(overwrite_existing=True)
await rag_conc.add_documents(docs)

await rag_conc.build_index_concurrent(
    index_type=IndexType.HNSW,
    m=16,
    ef_construction=64,
)
print("✅ Concurrent index built (zero downtime)")

# Check progress (useful for large datasets)
progress = await rag_conc.get_index_build_progress()
print(f"📊 Build progress: {progress}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Concurrent index built (zero downtime)
📊 Build progress: None


In [12]:
await rag_conc.delete_table()
await rag_conc.close()
print("🧹 All done!")

🧹 All done!
